# Banking AI-Agent — Colab runner

End-to-end runner for the Lab 3 banking agentic workflow. Run cells **in order, top to bottom**.

1. Install + start **Ollama**, pull the drafting LLM.
2. Clone the project repo and install dependencies (FastAPI + Unsloth).
3. Mount Google Drive and extract the **fine-tuned intent artifacts** from Lab 2.
4. Pre-flight VRAM check.
5. Launch the **FastAPI orchestrator** on port `8000`.
6. (Optional) Expose the port via Pinggy.
7. Demo the workflow on `examples/sample_requests.json`.

### Runtime requirement
Colab GPU runtime. **Tesla T4 (16 GB)** works with the default model below.

### Why we default to `qwen2.5:7b`
The lab handout *encourages* `gpt-oss:20b`, but that model needs ≥16 GB VRAM on its own — combined with the Lab 2 Unsloth-3B intent classifier (~2.5 GB) it OOMs a T4 (15.36 GB usable). On Colab Free the only reliable Ollama-served chat model is `qwen2.5:7b` (~4.7 GB) which still gives high-quality JSON-shaped replies and leaves comfortable headroom. If you have access to an A100/L4 just set `OLLAMA_MODEL = 'gpt-oss:20b'` in Cell 1 — the rest of the notebook adapts automatically.

### Critical do/don't on T4 (16 GB)
* **DO NOT** load `IntentClassification` directly inside this notebook kernel. The FastAPI server already loads its own copy; a second copy in the kernel doubles the VRAM cost and OOMs the GPU.
* If something goes wrong, run the **🛟 Troubleshoot** cell at the very bottom — it prints VRAM, processes, server log and listening ports in one go.


## Cell 1 — Install + start Ollama, pull the drafting LLM

Ollama is launched in a background thread inside this Colab kernel; FastAPI
will reach it over `localhost:11434`. We pull `qwen2.5:7b` by default — see
the top markdown for rationale.

In [1]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, threading, time

ollama_env = os.environ.copy()
ollama_env['OLLAMA_HOST']    = '0.0.0.0'
ollama_env['OLLAMA_ORIGINS'] = '*'
# Unload models 30 s after each request so the GPU is fully free between
# bursts of demo traffic. On T4 this is what makes coexisting with the
# Unsloth intent model reliable.
ollama_env['OLLAMA_KEEP_ALIVE'] = '30s'

def _serve():
    subprocess.Popen(['ollama', 'serve'], env=ollama_env)

threading.Thread(target=_serve, daemon=True).start()
time.sleep(6)
print('Ollama serve started in background.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package pci.ids.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacki

In [2]:
# Default = qwen2.5:7b (fits T4 + Unsloth 3B comfortably, ~7 GB peak).
# If you are on A100 / L4 (>= 24 GB VRAM) you can switch to gpt-oss:20b.
OLLAMA_MODEL = 'qwen2.5:7b'   # or 'gpt-oss:20b' on a bigger GPU

!ollama pull {OLLAMA_MODEL}
!curl -s http://localhost:11434/api/tags | head -c 600


{"models":[{"name":"qwen2.5:7b","model":"qwen2.5:7b","modified_at":"2026-05-11T19:41:28.614754191Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M"}}]}

## Cell 2 — Clone the repo and install Python deps

The intent model uses **Unsloth**, which has its own installer.

In [3]:
%cd /content
![ -d banking-ai-agent ] || git clone https://github.com/tzin1401/banking-ai-agent.git
%cd /content/banking-ai-agent
# Always pull the latest fixes (safe even on a fresh clone).
!git pull --rebase

/content
Cloning into 'banking-ai-agent'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 54 (delta 16), reused 50 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 40.32 KiB | 5.76 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/banking-ai-agent
Already up to date.


In [4]:
!pip install -q -r requirements.txt
# Unsloth needs its own installer on Colab:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

## Cell 3 — Mount Drive and extract the intent artifacts

The Lab 2 fine-tuned LoRA adapter lives in
`MyDrive/banking-ai-agent/intent_artifacts.zip` (~347 MB). After extraction
the project root contains `outputs/`, `sample_data/`, and `configs/inference.yaml`,
which is exactly what `IntentClassification` expects.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
ARTIFACT_ZIP = '/content/drive/MyDrive/banking-ai-agent/intent_artifacts.zip'
PROJECT_ROOT = '/content/banking-ai-agent'

import os, zipfile, shutil
assert os.path.exists(ARTIFACT_ZIP), f'Missing {ARTIFACT_ZIP} on Drive.'

# Skip extraction if already done (e.g. when re-running the notebook).
needs_extract = not all(
    os.path.exists(f'{PROJECT_ROOT}/{p}')
    for p in ('outputs/adapter_config.json', 'sample_data/labels.txt', 'configs/inference.yaml')
)

if needs_extract:
    print('Extracting artifacts...')
    with zipfile.ZipFile(ARTIFACT_ZIP) as zf:
        zf.extractall('/tmp/_intent')
    for sub in ('outputs', 'sample_data', 'configs'):
        src = f'/tmp/_intent/intent_artifacts/{sub}'
        dst = f'{PROJECT_ROOT}/{sub}'
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.move(src, dst)
    print('Done.')
else:
    print('Artifacts already in place — skipping extraction.')

!ls -lh /content/banking-ai-agent/outputs | head
!wc -l /content/banking-ai-agent/sample_data/labels.txt

Extracting artifacts...
Done.
total 388M
-rw-r--r-- 1 root root 1.2K May 11 19:42 adapter_config.json
-rw-r--r-- 1 root root 372M May 11 19:43 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K May 11 19:42 chat_template.jinja
-rw-r--r-- 1 root root 7.2K May 11 19:42 eval_results.txt
-rw-r--r-- 1 root root 1.6K May 11 19:42 README.md
-rw-r--r-- 1 root root  50K May 11 19:42 tokenizer_config.json
-rw-r--r-- 1 root root  17M May 11 19:42 tokenizer.json
77 /content/banking-ai-agent/sample_data/labels.txt


## Cell 4 — Pre-flight VRAM check (no model load)

Quick sanity check that the GPU is healthy and Ollama is serving the model.
**This cell deliberately does NOT load the intent classifier** — letting the
FastAPI server own the only copy keeps VRAM headroom on T4.

In [7]:
import requests

print('=== nvidia-smi ===')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

print('\n=== Ollama warm-up ping ===')
payload = {
    'model': OLLAMA_MODEL,
    'messages': [{'role': 'user', 'content': 'Reply with one word: ready.'}],
    'stream': False,
    'options': {'num_predict': 64, 'num_ctx': 4096},
}
# gpt-oss accepts reasoning_effort at the request top level.
if 'gpt-oss' in OLLAMA_MODEL:
    payload['reasoning_effort'] = 'low'
r = requests.post('http://localhost:11434/api/chat', json=payload, timeout=300)
print('content :', repr(r.json().get('message', {}).get('content', '')))

=== nvidia-smi ===
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB

=== Ollama warm-up ping ===
content : 'Ready.'


## Cell 5 — Launch the FastAPI server on port 8000

Starts `app.main:app` in the background with the right environment.
The first start can take ~30-60 s because the server loads the Unsloth LoRA
adapter at startup.

In [8]:
%cd /content/banking-ai-agent
import os, time, requests

# Make sure nothing else holds port 8000 before we start.
!fuser -k 8000/tcp 2>/dev/null
!pkill -f 'python run.py' 2>/dev/null
time.sleep(2)

os.environ['INTENT_MODE']          = 'unsloth'
os.environ['MOCK_LLM']             = '0'
os.environ['OLLAMA_URL']           = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']         = OLLAMA_MODEL
os.environ['INTENT_CONFIG_PATH']   = '/content/banking-ai-agent/configs/inference.yaml'
os.environ['INTENT_LABELS_PATH']   = '/content/banking-ai-agent/sample_data/labels.txt'
os.environ['PORT']                 = '8000'
# Wide budget for the LLM (room for either short replies or gpt-oss's
# chain-of-thought tokens) and a generous timeout.
os.environ['LLM_MAX_TOKENS']       = '2048'
os.environ['LLM_REASONING_EFFORT'] = 'low'    # only honoured by gpt-oss
os.environ['OLLAMA_TIMEOUT']       = '600'

!nohup python run.py > server.log 2>&1 &

# Wait up to 3 minutes for /health.
ok = False
for i in range(90):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok:
            ok = True
            print(f'\n✅ FastAPI ready after ~{i*2}s →', r.json())
            break
    except Exception:
        pass
    time.sleep(2)

if not ok:
    print('\n❌ Server did not come up. Tail of server.log:')
    !tail -50 server.log

/content/banking-ai-agent
^C

✅ FastAPI ready after ~90s → {'status': 'ok', 'intent_mode': 'unsloth', 'ollama_model': 'qwen2.5:7b', 'mock_llm': False}


In [9]:
# VRAM after Unsloth has loaded inside the FastAPI process.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
2703 MiB, 15360 MiB


## Cell 6 — (Optional) Open a Pinggy tunnel for external demo

> 💡 **You can skip this cell entirely.** Cell 7 already targets
> `http://localhost:8000` which works without any tunnel — perfect for the
> video demo.

Use Pinggy only if you want a **public URL** to call the agent from outside
Colab. Pinggy requires an interactive SSH session, which means a Colab
**terminal**. Colab Free does *not* expose a terminal; only Colab Pro does
(View → Terminal). With a terminal open, run:

```bash
ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io
```

Pinggy will print a real public URL of the form
`https://<random-id>.a.free.pinggy.link`. **Right-click → Copy** that URL
(`Ctrl+C` kills the tunnel) and paste it into the next cell.

> ⚠️ The string in the placeholder below is just a *format example* — do not
> paste it as-is, it will fail DNS resolution.

In [10]:
# OPTIONAL CELL — leave PUBLIC_URL empty to skip safely.
PUBLIC_URL = ''  # e.g. 'https://r4nDom-xx.a.free.pinggy.link' (use YOUR session's URL)

if PUBLIC_URL:
    import requests
    print(requests.get(f'{PUBLIC_URL}/health', timeout=10).json())
else:
    print('Skipping Pinggy probe: PUBLIC_URL is empty (use http://localhost:8000 instead).')

Skipping Pinggy probe: PUBLIC_URL is empty (use http://localhost:8000 instead).


## Cell 7 — Demo all sample requests

Loops over `examples/sample_requests.json` and pretty-prints the workflow
trace. Use the local URL (`http://localhost:8000`) or the public Pinggy URL
— both work.

* `qwen2.5:7b` typically returns each reply in **5-20 s**.
* `gpt-oss:20b` (if you switched) is slower (~30-90 s per reply) due to its chain-of-thought tokens.

Set `N` below to a small number first to smoke-test before sweeping all 10.

In [11]:
import json, time, textwrap, requests

TARGET = 'http://localhost:8000'   # or set to PUBLIC_URL for the tunnel
examples = json.load(open('/content/banking-ai-agent/examples/sample_requests.json'))

N = len(examples)   # change to e.g. 3 for a quick smoke check

ok = 0
for ex in examples[:N]:
    t0 = time.perf_counter()
    print('=' * 80)
    print('💬', ex['message'])
    try:
        r = requests.post(f'{TARGET}/process', json={'message': ex['message']}, timeout=600)
        data = r.json()
        action = data['decision']['action']
        match = 'OK' if action == ex['expected_action'] else 'DIFF'
        if match == 'OK':
            ok += 1
        print(f'   intent   : {data["trace"]["intent"]["intent"]}  ({data["trace"]["intent"]["source"]})')
        print(f'   priority : {data["trace"]["priority"]["level"]}')
        print(f'   policy   : {data["trace"]["policy"]["policy_id"]}')
        print(f'   action   : {action}   (expected {ex["expected_action"]}) [{match}]')
        print('   reply    :')
        print(textwrap.indent(textwrap.fill(data['final_response'], 70), '              '))
        if data['trace']['draft'].get('missing_info'):
            print(f'   missing  : {data["trace"]["draft"]["missing_info"]}')
        print(f'   latency  : {data["extra"].get("latency_ms")} ms  ({time.perf_counter()-t0:.1f}s wall)')
    except Exception as e:
        print(f'   ❌ error: {e}')

print(f'\n=== Decision match: {ok}/{N} ===')

💬 Hi, I ordered a new card 4 days ago but it still hasn't arrived. Can you check?
   intent   : card_arrival  (unsloth)
   priority : low
   policy   : card_arrival
   action   : reply   (expected reply) [OK]
   reply    :
              I understand your concern. It usually takes 5–7 working days for new
              cards to arrive. I will check on the status of your order right away.
   latency  : 33788.93 ms  (33.8s wall)
💬 I lost my card on the train today and I'm really worried someone might use it. Please help.
   intent   : lost_or_stolen_card  (unsloth)
   priority : high
   policy   : lost_or_stolen_card
   action   : escalate   (expected escalate) [OK]
   reply    :
              Thanks for your message. Your case has been escalated to one of our
              specialists — they will reach out to you shortly.
   latency  : 3144.09 ms  (3.2s wall)
💬 I sent a transfer to my friend yesterday but he says he still hasn't received the money.
   intent   : transfer_not_received_by_

## Cell 8 — Interactive query (optional)

Edit the message and re-run for the video demo.

In [12]:
import json, requests

msg = 'I lost my card on the bus this morning, please help!'
r = requests.post('http://localhost:8000/process', json={'message': msg}, timeout=600)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

{
  "final_response": "Thanks for your message. Your case has been escalated to one of our specialists — they will reach out to you shortly.",
  "decision": {
    "action": "escalate",
    "reason": "High priority issue: Detected urgent keywords in message: lost my card."
  },
  "trace": {
    "intent": {
      "intent": "lost_or_stolen_card",
      "confidence": 0.95,
      "source": "unsloth"
    },
    "priority": {
      "level": "high",
      "reason": "Detected urgent keywords in message: lost my card.",
      "matched_keywords": [
        "lost my card"
      ]
    },
    "policy": {
      "policy_id": "lost_or_stolen_card",
      "content": "Block the card immediately from the mobile app or by calling our\n24/7 hotline. Any transactions performed after the report time are\ncovered by our zero-liability policy. A replacement card is issued\nthe same working day and delivered within 5–7 working days.",
      "found": true
    },
    "draft": {
      "draft_reply": "I'm sorry to h

## 🛟 Troubleshoot — run this if anything misbehaves

Prints VRAM, listening port, server tail and Python processes in one go.
Useful when `/process` starts returning `Connection refused` or hanging.

In [13]:
print('=== nvidia-smi ===')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

print('\n=== Python processes ===')
!ps aux | grep -E 'python run\.py|ollama serve' | grep -v grep || echo '(no matching processes)'

print('\n=== Port 8000 ===')
!ss -tlnp 2>/dev/null | grep :8000 || echo 'Nothing on 8000.'

print('\n=== server.log (last 60 lines) ===')
!tail -60 /content/banking-ai-agent/server.log 2>/dev/null || echo '(no server.log)'

print('\n=== Ollama tags ===')
import requests
try:
    print(requests.get('http://localhost:11434/api/tags', timeout=5).json())
except Exception as e:
    print('Ollama unreachable:', e)

=== nvidia-smi ===
memory.used [MiB], memory.total [MiB]
8363 MiB, 15360 MiB

=== Python processes ===
root        5012  8.0  0.8 2475208 116368 ?      Sl   19:40   0:33 ollama serve

=== Port 8000 ===
LISTEN 0      2048         0.0.0.0:8000       0.0.0.0:*    users:(("python3",pid=6540,fd=56))     

=== server.log (last 60 lines) ===
INFO:     Started server process [6540]
INFO:     Waiting for application startup.
2026-05-11 19:45:05.815569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
   Loaded 77 valid labels from ./sample_data/labels.txt
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers:

### Common failure modes and fixes

* **VRAM ~0 MiB + no Python processes** → both Ollama and FastAPI died (usually CUDA OOM). **Solution:** Runtime → Restart runtime, then re-run from Cell 1 with `OLLAMA_MODEL = 'qwen2.5:7b'`.
* **Ollama tags works, but server.log says `Address already in use`** → an old server is squatting on port 8000. Re-run Cell 5 (it kills the old process automatically).
* **server.log shows `Empty content AND empty thinking`** → `LLM_MAX_TOKENS` is too small. Raise it (`4096`) and re-run Cell 5.
* **`ollama pull` fails with HTTP 404** → the tag you typed doesn't exist in the Ollama library. `gpt-oss` only ships in `:20b` and `:120b` sizes; for T4 use `qwen2.5:7b` (default).